# FastAPI: 4 Hands-on Tutorials

A progressive, Google Colab-ready course from your first endpoint to a tested mini-project.

## Learning path
1. **Fundamentals** — routes, path parameters, query parameters, and automatic documentation
2. **Validated CRUD API** — Pydantic models, status codes, and in-memory data
3. **Security and errors** — API-key authentication, dependencies, and exception handling
4. **Mini-project** — support-ticket API with filters, pagination, metrics, and tests

> Colab note: FastAPI is an ASGI web application. The notebook tests it directly using `TestClient`, so every exercise works without opening a server. The final section also shows how to run a public demo URL.

## 0. Environment setup

In [ ]:
%pip install -q fastapi uvicorn httpx pyngrok

from fastapi import FastAPI, HTTPException, Depends, Header, Query, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import Literal
from datetime import datetime, timezone
from uuid import uuid4

print('FastAPI environment is ready.')

# Tutorial 1 — FastAPI Fundamentals

### Goal
Build a small course-catalogue API and learn the three most common input styles:
- Route: `GET /`
- Path parameter: `GET /courses/{course_id}`
- Query parameters: `GET /courses?level=beginner&limit=2`

In [ ]:
app1 = FastAPI(title='Course Catalogue API', version='1.0')

courses = [
    {'id': 1, 'title': 'Python Basics', 'level': 'beginner'},
    {'id': 2, 'title': 'FastAPI Fundamentals', 'level': 'beginner'},
    {'id': 3, 'title': 'Agentic AI with LangGraph', 'level': 'advanced'},
]

@app1.get('/')
def home():
    return {'message': 'Welcome to the Course Catalogue API'}

@app1.get('/courses')
def list_courses(level: str | None = None, limit: int = Query(default=10, ge=1, le=100)):
    result = courses
    if level:
        result = [course for course in result if course['level'].lower() == level.lower()]
    return {'count': min(len(result), limit), 'items': result[:limit]}

@app1.get('/courses/{course_id}')
def get_course(course_id: int):
    for course in courses:
        if course['id'] == course_id:
            return course
    raise HTTPException(status_code=404, detail='Course not found')

In [ ]:
client1 = TestClient(app1)

print('Home:', client1.get('/').json())
print('All:', client1.get('/courses').json())
print('Filtered:', client1.get('/courses', params={'level': 'beginner', 'limit': 1}).json())
print('One course:', client1.get('/courses/2').json())
print('Missing:', client1.get('/courses/999').status_code, client1.get('/courses/999').json())

### Exercise 1
Add `GET /courses/search/{keyword}`. It should return courses whose titles contain the keyword, ignoring uppercase/lowercase.

**Important:** define this route before `/courses/{course_id}` in a production application, because route order can affect matching.

In [ ]:
# Suggested solution (try it yourself first)
@app1.get('/search-courses/{keyword}')
def search_courses(keyword: str):
    matches = [c for c in courses if keyword.lower() in c['title'].lower()]
    return {'count': len(matches), 'items': matches}

assert client1.get('/search-courses/api').json()['count'] == 1
print(client1.get('/search-courses/api').json())

# Tutorial 2 — Validated CRUD API

### Goal
Build Create, Read, Update, and Delete operations. Pydantic validates request bodies and also documents their schema automatically. Data is stored in memory, so it resets when the runtime restarts.

In [ ]:
class ProductCreate(BaseModel):
    name: str = Field(min_length=2, max_length=80)
    price: float = Field(gt=0)
    stock: int = Field(default=0, ge=0)
    category: str = Field(min_length=2, max_length=40)

class Product(ProductCreate):
    id: int

app2 = FastAPI(title='Product CRUD API')
product_db: dict[int, Product] = {}
next_product_id = 1

@app2.post('/products', response_model=Product, status_code=status.HTTP_201_CREATED)
def create_product(payload: ProductCreate):
    global next_product_id
    product = Product(id=next_product_id, **payload.model_dump())
    product_db[next_product_id] = product
    next_product_id += 1
    return product

@app2.get('/products', response_model=list[Product])
def list_products():
    return list(product_db.values())

@app2.get('/products/{product_id}', response_model=Product)
def get_product(product_id: int):
    if product_id not in product_db:
        raise HTTPException(404, 'Product not found')
    return product_db[product_id]

@app2.put('/products/{product_id}', response_model=Product)
def replace_product(product_id: int, payload: ProductCreate):
    if product_id not in product_db:
        raise HTTPException(404, 'Product not found')
    product_db[product_id] = Product(id=product_id, **payload.model_dump())
    return product_db[product_id]

@app2.delete('/products/{product_id}', status_code=status.HTTP_204_NO_CONTENT)
def delete_product(product_id: int):
    if product_id not in product_db:
        raise HTTPException(404, 'Product not found')
    del product_db[product_id]
    return None

In [ ]:
client2 = TestClient(app2)
new_product = {'name': 'Wireless Mouse', 'price': 999.0, 'stock': 25, 'category': 'Electronics'}
created = client2.post('/products', json=new_product)
print('CREATE:', created.status_code, created.json())
print('READ:', client2.get('/products/1').json())
updated = {**new_product, 'price': 899.0, 'stock': 20}
print('UPDATE:', client2.put('/products/1', json=updated).json())
invalid = client2.post('/products', json={**new_product, 'price': -1})
print('VALIDATION:', invalid.status_code, invalid.json()['detail'][0]['msg'])
print('DELETE:', client2.delete('/products/1').status_code)

### Exercise 2
Add a partial-update endpoint using `PATCH`. Unlike `PUT`, omitted fields should remain unchanged.

Hint: create a model whose fields all default to `None`, then use `model_dump(exclude_unset=True)`.

In [ ]:
class ProductPatch(BaseModel):
    name: str | None = Field(default=None, min_length=2, max_length=80)
    price: float | None = Field(default=None, gt=0)
    stock: int | None = Field(default=None, ge=0)
    category: str | None = Field(default=None, min_length=2, max_length=40)

@app2.patch('/products/{product_id}', response_model=Product)
def patch_product(product_id: int, payload: ProductPatch):
    if product_id not in product_db:
        raise HTTPException(404, 'Product not found')
    changes = payload.model_dump(exclude_unset=True)
    updated_data = product_db[product_id].model_dump() | changes
    product_db[product_id] = Product(**updated_data)
    return product_db[product_id]

client2.post('/products', json=new_product)
print(client2.patch('/products/2', json={'stock': 50}).json())

# Tutorial 3 — Authentication, Dependencies, and Errors

### Goal
Protect routes with an API key. FastAPI's dependency system keeps reusable concerns—authentication, database sessions, logging—outside endpoint logic.

> This API-key example is educational. Production systems should store secrets outside source code, use HTTPS, rotate keys, and normally use OAuth2/JWT or an identity provider.

In [ ]:
app3 = FastAPI(title='Secured Notes API')
DEMO_API_KEY = 'fastapi-demo-key'
notes: dict[int, dict] = {1: {'id': 1, 'text': 'Learn dependencies'}}

def verify_api_key(x_api_key: str | None = Header(default=None)):
    if x_api_key is None:
        raise HTTPException(status_code=401, detail='API key is required')
    if x_api_key != DEMO_API_KEY:
        raise HTTPException(status_code=403, detail='Invalid API key')
    return x_api_key

@app3.get('/health')
def health():
    return {'status': 'healthy'}

@app3.get('/notes', dependencies=[Depends(verify_api_key)])
def list_notes():
    return list(notes.values())

@app3.get('/notes/{note_id}')
def get_note(note_id: int, api_key: str = Depends(verify_api_key)):
    if note_id not in notes:
        raise HTTPException(404, detail={'message': 'Note not found', 'note_id': note_id})
    return notes[note_id]

In [ ]:
client3 = TestClient(app3)
print('Public:', client3.get('/health').status_code)
print('No key:', client3.get('/notes').status_code, client3.get('/notes').json())
print('Bad key:', client3.get('/notes', headers={'X-API-Key': 'wrong'}).status_code)
good_headers = {'X-API-Key': DEMO_API_KEY}
print('Valid key:', client3.get('/notes', headers=good_headers).status_code, client3.get('/notes', headers=good_headers).json())

### Exercise 3
Create `DELETE /notes/{note_id}`, protect it with the same dependency, return `204` on success, and return `404` when the note does not exist.

In [ ]:
@app3.delete('/notes/{note_id}', status_code=204, dependencies=[Depends(verify_api_key)])
def delete_note(note_id: int):
    if note_id not in notes:
        raise HTTPException(404, 'Note not found')
    del notes[note_id]
    return None

response = client3.delete('/notes/1', headers=good_headers)
assert response.status_code == 204
print('Protected delete passed.')

# Tutorial 4 — Mini-project: Support Ticket API

### Scenario
An IT support team needs an API to create tickets, list them with filters and pagination, update status, and view a small dashboard.

### Concepts combined
Pydantic validation, enums through `Literal`, CRUD, filtering, pagination, status codes, timestamps, and automated tests.

In [ ]:
Priority = Literal['low', 'medium', 'high', 'critical']
TicketStatus = Literal['open', 'in_progress', 'resolved']

class TicketCreate(BaseModel):
    title: str = Field(min_length=5, max_length=120)
    description: str = Field(min_length=10, max_length=1000)
    priority: Priority = 'medium'
    module: str = Field(min_length=2, max_length=30)

class Ticket(BaseModel):
    id: str
    title: str
    description: str
    priority: Priority
    module: str
    status: TicketStatus
    created_at: datetime

class StatusUpdate(BaseModel):
    status: TicketStatus

app4 = FastAPI(title='IT Support Ticket API', version='1.0.0')
ticket_db: dict[str, Ticket] = {}

@app4.post('/tickets', response_model=Ticket, status_code=201)
def create_ticket(payload: TicketCreate):
    ticket_id = str(uuid4())
    ticket = Ticket(
        id=ticket_id, **payload.model_dump(), status='open',
        created_at=datetime.now(timezone.utc)
    )
    ticket_db[ticket_id] = ticket
    return ticket

@app4.get('/tickets', response_model=list[Ticket])
def list_tickets(
    ticket_status: TicketStatus | None = Query(default=None, alias='status'),
    priority: Priority | None = None,
    module: str | None = None,
    skip: int = Query(default=0, ge=0),
    limit: int = Query(default=10, ge=1, le=100),
):
    result = list(ticket_db.values())
    if ticket_status:
        result = [t for t in result if t.status == ticket_status]
    if priority:
        result = [t for t in result if t.priority == priority]
    if module:
        result = [t for t in result if t.module.lower() == module.lower()]
    return result[skip:skip + limit]

@app4.get('/tickets/{ticket_id}', response_model=Ticket)
def get_ticket(ticket_id: str):
    if ticket_id not in ticket_db:
        raise HTTPException(404, 'Ticket not found')
    return ticket_db[ticket_id]

@app4.patch('/tickets/{ticket_id}/status', response_model=Ticket)
def update_ticket_status(ticket_id: str, payload: StatusUpdate):
    if ticket_id not in ticket_db:
        raise HTTPException(404, 'Ticket not found')
    ticket_db[ticket_id] = ticket_db[ticket_id].model_copy(update={'status': payload.status})
    return ticket_db[ticket_id]

@app4.get('/dashboard/summary')
def dashboard_summary():
    tickets = list(ticket_db.values())
    return {
        'total': len(tickets),
        'by_status': {s: sum(t.status == s for t in tickets) for s in ['open', 'in_progress', 'resolved']},
        'critical_open': sum(t.priority == 'critical' and t.status == 'open' for t in tickets),
    }

In [ ]:
client4 = TestClient(app4)
samples = [
    {'title': 'SAP login is failing', 'description': 'User receives an authentication error.', 'priority': 'high', 'module': 'BTP'},
    {'title': 'Production integration stopped', 'description': 'Messages are stuck in the integration queue.', 'priority': 'critical', 'module': 'CPI'},
    {'title': 'Incorrect purchase order total', 'description': 'Tax is calculated twice in the purchase order.', 'priority': 'medium', 'module': 'MM'},
]
created_tickets = [client4.post('/tickets', json=item).json() for item in samples]
ticket_id = created_tickets[0]['id']
print('Created:', len(created_tickets))
print('High priority:', client4.get('/tickets', params={'priority': 'high'}).json())
print('Status update:', client4.patch(f'/tickets/{ticket_id}/status', json={'status': 'in_progress'}).json()['status'])
print('Dashboard:', client4.get('/dashboard/summary').json())

## Automated API tests
These assertions are small regression tests. If an assertion fails, Colab shows the exact failing cell.

In [ ]:
def run_api_tests():
    test_client = TestClient(app4)
    valid = {
        'title': 'Unable to approve workflow',
        'description': 'The approval button remains disabled for the manager.',
        'priority': 'high',
        'module': 'BPA',
    }
    create_response = test_client.post('/tickets', json=valid)
    assert create_response.status_code == 201
    new_id = create_response.json()['id']
    assert test_client.get(f'/tickets/{new_id}').status_code == 200
    assert test_client.get('/tickets/not-a-real-id').status_code == 404
    assert test_client.post('/tickets', json={**valid, 'priority': 'urgent'}).status_code == 422
    assert test_client.patch(f'/tickets/{new_id}/status', json={'status': 'resolved'}).json()['status'] == 'resolved'
    assert test_client.get('/tickets', params={'limit': 0}).status_code == 422
    print('✅ All API tests passed.')

run_api_tests()

### Exercise 4 — Extend the project
Implement one or more enhancements:
1. Add `DELETE /tickets/{ticket_id}`.
2. Add free-text search across title and description.
3. Prevent a resolved ticket from moving back to `open`.
4. Replace the dictionary with SQLite and SQLAlchemy/SQLModel.
5. Add JWT authentication and restrict status updates to support agents.

# Optional — Run the API from Google Colab

FastAPI normally runs with Uvicorn. In Colab, use a background thread and an ngrok tunnel. You need a free ngrok account and auth token. Never share or hard-code the token in a notebook.

After running the cells, open `{PUBLIC_URL}/docs` for Swagger UI or `{PUBLIC_URL}/redoc` for ReDoc.

In [ ]:
# Run only if you want a public demo URL.
# from getpass import getpass
# from pyngrok import ngrok
# import uvicorn, threading
#
# ngrok.set_auth_token(getpass('Enter your ngrok auth token: '))
# server = uvicorn.Server(uvicorn.Config(app4, host='0.0.0.0', port=8000, log_level='warning'))
# thread = threading.Thread(target=server.run, daemon=True)
# thread.start()
# public_url = ngrok.connect(8000).public_url
# print('API:', public_url)
# print('Swagger UI:', public_url + '/docs')

# Quick reference

| Need | FastAPI feature |
|---|---|
| Read data | `@app.get(...)` |
| Create data | `@app.post(...)` |
| Full replacement | `@app.put(...)` |
| Partial update | `@app.patch(...)` |
| Delete data | `@app.delete(...)` |
| Validate a body | A parameter typed as a Pydantic `BaseModel` |
| Validate query input | `Query(...)` |
| Return an API error | `raise HTTPException(...)` |
| Reuse auth or other logic | `Depends(...)` |
| Interactive docs | `/docs` |
| Alternative docs | `/redoc` |
| OpenAPI schema | `/openapi.json` |

## Recommended next steps
Learn asynchronous endpoints, SQLModel/SQLAlchemy, Alembic migrations, OAuth2/JWT, environment-based configuration, Docker, logging, and deployment.